# Pipeline Complet - Fine-tuning Mistral 7B sur Données Publiques Françaises

Ce notebook reproduit l'ensemble du pipeline du projet ADS :
1. **Installation** des dépendances
2. **Collecte** des données publiques (PIAF, DECP, Budgets, Délibérations)
3. **Extraction** et filtrage DECP (9 départements Sud France)
4. **Préparation** du corpus Q/A
5. **Fine-tuning** QLoRA de Mistral 7B v0.3
6. **Évaluation** (perplexité, accuracy, guardrails, vitesse)
7. **Test interactif** du modèle

**Matériel requis** : GPU NVIDIA avec >= 12 GB VRAM (ex: RTX 4070 Ti)  
**Temps estimé** : ~2h (collecte ~30min, préparation ~15min, entraînement ~90min)

## 1. Installation des dépendances

In [ ]:
# Installation des dependances (ajustez cu118 selon votre version CUDA : cu118, cu121, cu124)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.36.0 peft>=0.7.0 bitsandbytes>=0.41.0 huggingface_hub
!pip install datasets accelerate sentencepiece protobuf
!pip install requests pandas numpy tqdm

## 2. Configuration globale

In [1]:
import os
import json
import hashlib
import requests
import time
import torch
import numpy as np
from pathlib import Path
from datetime import datetime

# Auto-detection de la racine du projet
# Fonctionne que le CWD soit la racine du workspace OU le dossier du notebook
_cwd = Path(".").resolve()
PROJECT_ROOT = None
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / "code").exists() and (_candidate / "data").exists() and (_candidate / "requirements.txt").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        f"Impossible de trouver la racine du projet depuis {_cwd}. "
        f"Lancez le notebook depuis le repo ADS."
    )

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FT_DIR = DATA_DIR / "fine_tuning"
MODEL_DIR = PROJECT_ROOT / "models"
ADAPTERS_DIR = MODEL_DIR / "adapters"
RESULTS_DIR = PROJECT_ROOT / "results"

for d in [RAW_DIR, PROCESSED_DIR, FT_DIR, ADAPTERS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configuration du modele
MODEL_NAME = "mistralai/Mistral-7B-v0.3"

# System prompt specialise
SYSTEM_PROMPT = """Tu es un assistant spécialisé dans l'accès aux données publiques françaises, notamment :
- DECP (Données Essentielles de la Commande Publique)
- RNE (Répertoire National des Élus)

Tu réponds avec précision en citant tes sources. Si une information n'est pas dans ton corpus, tu le dis clairement."""

print(f"Projet : {PROJECT_ROOT}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Projet : D:\Projets\ADS
GPU : NVIDIA GeForce RTX 4070 Ti
VRAM : 12.0 GB


In [ ]:
# --- Authentification HuggingFace ---
# Mistral 7B v0.3 est un modele a acces restreint (gated model).
# Prerequis :
#   1. Creer un compte sur https://huggingface.co
#   2. Accepter la licence sur https://huggingface.co/mistralai/Mistral-7B-v0.3
#   3. Creer un token (READ) sur https://huggingface.co/settings/tokens
#   4. Definir HF_TOKEN en variable d'environnement OU coller le token ci-dessous

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Authentifie via variable d'environnement HF_TOKEN")
else:
    try:
        from huggingface_hub import HfApi
        user = HfApi().whoami()
        print(f"Deja authentifie comme : {user['name']}")
    except Exception:
        print("Non authentifie. Collez votre token HuggingFace ci-dessous.")
        print("(Creez-en un sur https://huggingface.co/settings/tokens)")
        token = input("Token HF : ")
        if token.strip():
            login(token=token.strip())
            print("Authentification reussie !")
        else:
            print("ATTENTION : pas de token fourni. Le telechargement de Mistral 7B echouera.")

Non authentifie. Collez votre token HuggingFace ci-dessous.
(Creez-en un sur https://huggingface.co/settings/tokens)


## 3. Collecte des données publiques

Sources utilisées :
- **PIAF** : Dataset Q/R français (HuggingFace) - ~3,835 exemples
- **DECP** : Marchés publics (data.gouv.fr) - **fichiers JSON mensuels** (~2-7 MB chacun, dataset `68caf6b135f19236a4f37a32`)
- **Budgets** : Balances comptables communales 2023 (API data.economie.gouv.fr)
- **Délibérations** : Actes municipaux SCDL (data.gouv.fr)

In [2]:
# --- 3.1 PIAF (Questions/Réponses en français) ---
from datasets import load_dataset

piaf_dir = RAW_DIR / "piaf"
piaf_dir.mkdir(parents=True, exist_ok=True)
piaf_file = piaf_dir / "piaf_train.jsonl"

if piaf_file.exists():
    print(f"PIAF deja telecharge : {piaf_file.stat().st_size / 1024**2:.1f} MB")
else:
    print("Telechargement PIAF depuis HuggingFace...")
    dataset = load_dataset("piaf")
    with open(piaf_file, 'w', encoding='utf-8') as f:
        for ex in dataset["train"]:
            entry = {
                "prompt": f"Question : {ex['question']}\nContexte : {ex['context'][:500]}...",
                "completion": ex['answers']['text'][0] if ex['answers']['text'] else "",
                "source": "PIAF"
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"PIAF : {len(dataset['train'])} exemples | {piaf_file.stat().st_size / 1024**2:.1f} MB")

d:\Projets\ADS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Telechargement PIAF depuis HuggingFace...


PIAF : 3835 exemples | 2.5 MB


In [3]:
# --- 3.2 DECP (Marchés publics - fichiers mensuels data.gouv.fr) ---
# Source : https://www.data.gouv.fr/fr/datasets/68caf6b135f19236a4f37a32/
# Les fichiers mensuels JSON (~2-7 MB chacun) remplacent le fichier XML consolidé (~600 MB)

decp_dir = RAW_DIR / "decp"
decp_dir.mkdir(parents=True, exist_ok=True)
decp_monthly_dir = decp_dir / "monthly"
decp_monthly_dir.mkdir(parents=True, exist_ok=True)

DATASET_ID = "68caf6b135f19236a4f37a32"
N_FILES = 4  # Nombre de fichiers mensuels à télécharger (ajuster selon besoins)

existing = list(decp_monthly_dir.glob("*.json"))
if len(existing) >= N_FILES:
    print(f"DECP déjà téléchargé : {len(existing)} fichiers mensuels dans {decp_monthly_dir}")
else:
    print("Récupération de la liste des fichiers DECP mensuels sur data.gouv.fr...")
    resp = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{DATASET_ID}/", timeout=15)
    resp.raise_for_status()
    resources = resp.json().get("resources", [])

    # Filtrer : JSON 2024 ou 2025, trier par titre décroissant, prendre les N_FILES plus récents
    json_res = sorted(
        [r for r in resources
         if r.get("format", "").lower() == "json"
         and any(y in r.get("title", "") for y in ["2024", "2025"])],
        key=lambda r: r.get("title", ""),
        reverse=True
    )[:N_FILES]

    print(f"Téléchargement de {len(json_res)} fichiers DECP mensuels…")
    for res in json_res:
        title = res.get("title", "unknown")
        url = res.get("url")
        dest = decp_monthly_dir / f"{title}"
        if dest.exists():
            print(f"  {title} : déjà présent ({dest.stat().st_size / 1024**2:.1f} MB)")
            continue
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        dest.write_bytes(r.content)
        print(f"  {title} : {dest.stat().st_size / 1024**2:.1f} MB")
        time.sleep(0.5)

    print(f"DECP mensuel téléchargé ({len(list(decp_monthly_dir.glob('*.json')))} fichiers)")


Récupération de la liste des fichiers DECP mensuels sur data.gouv.fr...
Téléchargement de 4 fichiers DECP mensuels…
  donneesEssentielles_2024-01-05.json : 0.5 MB
  aws_2025-12.json : 4.9 MB
  aws_2025-11.json : 3.1 MB
  aws_2025-03.json : 3.0 MB
DECP mensuel téléchargé (4 fichiers)


In [4]:
# --- 3.3 Budgets municipaux 2023 ---
budgets_dir = RAW_DIR / "budgets"
budgets_dir.mkdir(parents=True, exist_ok=True)

base_url = "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/balances-comptables-des-communes-en-2023/records"
cities = ["PARIS", "LYON", "MARSEILLE", "TOULOUSE", "NICE",
          "NANTES", "MONTPELLIER", "STRASBOURG", "BORDEAUX", "LILLE"]

total_records = 0
for city in cities:
    city_file = budgets_dir / f"{city.lower()}_2023.json"
    if city_file.exists():
        total_records += len(json.loads(city_file.read_text(encoding='utf-8')))
        continue
    try:
        params = {"where": f'lbudg like "{city}%"', "limit": 100}
        resp = requests.get(base_url, params=params, timeout=30)
        resp.raise_for_status()
        records = resp.json().get("results", [])
        if records:
            with open(city_file, 'w', encoding='utf-8') as f:
                json.dump(records, f, indent=2, ensure_ascii=False)
            total_records += len(records)
            print(f"{city}: {len(records)} lignes")
        time.sleep(0.5)
    except Exception as e:
        print(f"{city}: Erreur - {e}")

print(f"\nTotal budgets : {total_records} lignes")

PARIS: 100 lignes
LYON: 100 lignes
MARSEILLE: 100 lignes
TOULOUSE: 100 lignes
NICE: 100 lignes
NANTES: 100 lignes
MONTPELLIER: 100 lignes
STRASBOURG: 100 lignes
BORDEAUX: 100 lignes
LILLE: 100 lignes

Total budgets : 1000 lignes


In [5]:
# --- 3.4 Deliberations municipales ---
delib_dir = RAW_DIR / "deliberations"
delib_dir.mkdir(parents=True, exist_ok=True)

search_url = "https://www.data.gouv.fr/api/1/datasets/"
params = {"q": "deliberations SCDL", "page_size": 20}

try:
    resp = requests.get(search_url, params=params, timeout=30)
    resp.raise_for_status()
    datasets_found = resp.json().get("data", [])
    print(f"Trouve {len(datasets_found)} datasets de deliberations")

    collected = 0
    for ds in datasets_found[:5]:
        for resource in ds.get("resources", [])[:2]:
            fmt = resource.get("format", "").lower()
            if fmt in ["csv", "json"] and collected < 5:
                try:
                    url = resource.get("url")
                    r = requests.get(url, timeout=60, stream=True)
                    r.raise_for_status()
                    cl = int(r.headers.get('content-length', 0))
                    if cl > 15 * 1024 * 1024:
                        continue
                    filepath = delib_dir / f"delib_{collected}.{fmt}"
                    with open(filepath, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            f.write(chunk)
                    print(f"  delib_{collected}.{fmt}: {filepath.stat().st_size/1024**2:.1f} MB")
                    collected += 1
                    time.sleep(1)
                except Exception as e:
                    print(f"  Erreur telechargement: {e}")
    print(f"Deliberations : {collected} fichiers")
except Exception as e:
    print(f"Erreur : {e}")

Trouve 2 datasets de deliberations
  delib_0.csv: 0.4 MB
  delib_1.json: 0.9 MB
  delib_2.csv: 1.6 MB
  delib_3.json: 3.7 MB
Deliberations : 4 fichiers


## 4. Extraction et filtrage DECP

Filtre les marchés publics par :
- **Zone** : 9 départements du Sud de la France
- **Période** : 2023-2025

In [6]:
REGIONS_CIBLES = {
    "HERAULT-34":            ["34"],
    "HAUTE-GARONNE-31":      ["31"],
    "GARD-30":               ["30"],
    "TARN-81":               ["81"],
    "BOUCHES-DU-RHONE-13":   ["13"],
    "RHONE-69":              ["69"],
    "GIRONDE-33":            ["33"],
    "PYRENEES-ORIENTALES-66":["66"],
    "AUDE-11":               ["11"],
}

decp_filtered_file = PROCESSED_DIR / "decp_filtered.jsonl"

if decp_filtered_file.exists():
    with open(decp_filtered_file, encoding='utf-8') as _f:
        n_lines = sum(1 for _ in _f)
    print(f"DECP déjà filtré : {n_lines} marchés dans {decp_filtered_file}")
else:
    marches = []
    stats = {v: 0 for v in REGIONS_CIBLES}

    json_files = sorted(decp_monthly_dir.glob("*.json"))
    print(f"Extraction depuis {len(json_files)} fichier(s) mensuel(s)…")

    for json_file in json_files:
        try:
            with open(json_file, encoding='utf-8') as f:
                data = json.load(f)

            # Format : {"marches": {"marche": [...]}}
            marches_data = data.get("marches", {})
            if isinstance(marches_data, dict):
                marches_list = marches_data.get("marche", [])
            elif isinstance(marches_data, list):
                marches_list = marches_data
            else:
                marches_list = []

            count = 0
            for rec in marches_list:
                lieu = rec.get("lieuExecution", {})
                code_postal = str(lieu.get("code", "")) if isinstance(lieu, dict) else ""

                dept = None
                for region, codes in REGIONS_CIBLES.items():
                    if any(code_postal.startswith(c) for c in codes):
                        dept = region
                        break
                if not dept:
                    continue

                acheteur_info = rec.get("acheteur", {})
                acheteur_nom = (
                    acheteur_info.get("nom", "Non renseigné")
                    if isinstance(acheteur_info, dict)
                    else str(acheteur_info or "Non renseigné")
                )
                titulaires = rec.get("titulaires", [])
                titulaire_nom = (
                    titulaires[0].get("denominationSociale", "")
                    if titulaires and isinstance(titulaires, list)
                    else ""
                )

                marches.append({
                    'departement': dept,
                    'acheteur': acheteur_nom,
                    'objet': str(rec.get('objet', '')),
                    'montant': rec.get('montant', '0'),
                    'nature': str(rec.get('nature', 'Marché')),
                    'duree': str(rec.get('dureeMois', '')),
                    'procedure': str(rec.get('procedure', '')),
                    'cpv': str(rec.get('codeCPV', '')),
                    'date': str(rec.get('dateNotification', ''))[:10],
                    'titulaire': titulaire_nom,
                })
                stats[dept] += 1
                count += 1

            print(f"  {json_file.name} : {count} marchés Sud France extraits (sur {len(marches_list)} total)")

        except Exception as e:
            print(f"  Erreur {json_file.name} : {e}")

    with open(decp_filtered_file, 'w', encoding='utf-8') as f:
        for m in marches:
            f.write(json.dumps(m, ensure_ascii=False) + '\n')

    print(f"\nTotal extrait : {len(marches):,} marchés")
    for dept, count in stats.items():
        if count > 0:
            print(f"  {dept}: {count:,}")


Extraction depuis 4 fichier(s) mensuel(s)…
  aws_2025-03.json : 532 marchés Sud France extraits (sur 2642 total)
  aws_2025-11.json : 565 marchés Sud France extraits (sur 2757 total)
  aws_2025-12.json : 991 marchés Sud France extraits (sur 4350 total)
  donneesEssentielles_2024-01-05.json : 140 marchés Sud France extraits (sur 578 total)

Total extrait : 2,228 marchés
  HERAULT-34: 351
  HAUTE-GARONNE-31: 391
  GARD-30: 133
  TARN-81: 49
  BOUCHES-DU-RHONE-13: 411
  RHONE-69: 379
  GIRONDE-33: 330
  PYRENEES-ORIENTALES-66: 142
  AUDE-11: 42


## 5. Preparation du corpus de fine-tuning

Genere des paires Question/Reponse a partir des donnees collectees, puis nettoie et fusionne le tout en un corpus optimise pour 12 GB VRAM.

In [7]:
import random
random.seed(42)

# --- 5.1 Generation Q/A a partir des marches DECP ---
qa_pairs = []

# Charger marches filtres
marches_data = []
with open(decp_filtered_file, 'r', encoding='utf-8') as f:
    for line in f:
        marches_data.append(json.loads(line))

print(f"Marches charges : {len(marches_data)}")

# Templates de questions pour diversite
QUESTION_TEMPLATES = [
    ("montant", [
        "Quel est le montant du marche '{objet}' passe par {acheteur} ?",
        "Combien coute le marche de {acheteur} concernant '{objet}' ?",
        "A combien s'eleve le marche '{objet}' dans le departement {departement} ?"
    ]),
    ("objet", [
        "Quel est l'objet du marche passe par {acheteur} le {date} ?",
        "Decrivez le marche public attribue par {acheteur} en {departement}."
    ]),
    ("procedure", [
        "Quelle procedure a ete utilisee pour le marche '{objet}' ?",
        "Comment le marche de {acheteur} a-t-il ete attribue ?"
    ]),
    ("acheteur", [
        "Qui a passe le marche '{objet}' dans le departement {departement} ?",
        "Quel organisme public a lance l'appel d'offres pour '{objet}' ?"
    ])
]

for marche in marches_data:
    for q_type, templates in QUESTION_TEMPLATES:
        template = random.choice(templates)
        try:
            question = template.format(**marche)
        except KeyError:
            continue
        
        # Construire reponse selon le type
        if q_type == "montant":
            montant = marche.get('montant', '0')
            reponse = (f"Le marche '{marche['objet']}' passe par {marche['acheteur']} "
                      f"a un montant de {montant} EUR. "
                      f"[Source : DECP {marche['departement']} - {marche['date']}]")
        elif q_type == "objet":
            reponse = (f"Le marche attribue par {marche['acheteur']} porte sur : "
                      f"{marche['objet']}. Montant : {marche.get('montant', 'NC')} EUR. "
                      f"[Source : DECP {marche['departement']}]")
        elif q_type == "procedure":
            proc = marche.get('procedure', 'Non renseignee')
            reponse = (f"Le marche '{marche['objet']}' a ete attribue via la procedure : {proc}. "
                      f"[Source : DECP {marche['departement']}]")
        elif q_type == "acheteur":
            reponse = (f"Le marche '{marche['objet']}' a ete passe par {marche['acheteur']} "
                      f"dans le departement {marche['departement']}. "
                      f"[Source : DECP donnees reelles]")
        
        qa_pairs.append({"prompt": question, "completion": reponse, "source": "DECP"})

print(f"Paires Q/A generees depuis DECP : {len(qa_pairs)}")

Marches charges : 2228
Paires Q/A generees depuis DECP : 8912


In [8]:
# --- 5.2 Ajouter les paires PIAF ---
piaf_pairs = []
if piaf_file.exists():
    with open(piaf_file, 'r', encoding='utf-8') as f:
        for line in f:
            piaf_pairs.append(json.loads(line))
    print(f"Paires PIAF : {len(piaf_pairs)}")

# --- 5.3 Fusion et deduplication ---
all_pairs = qa_pairs + piaf_pairs
print(f"Total avant dedup : {len(all_pairs)}")

# Deduplication par hash SHA-256
seen = set()
unique_pairs = []
for pair in all_pairs:
    h = hashlib.sha256((pair['prompt'] + pair['completion']).encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique_pairs.append(pair)

print(f"Apres deduplication : {len(unique_pairs)}")
print(f"Doublons supprimes : {len(all_pairs) - len(unique_pairs)}")

Paires PIAF : 3835
Total avant dedup : 12747
Apres deduplication : 10973
Doublons supprimes : 1774


In [9]:
# --- 5.4 Optimisation pour 12 GB VRAM ---
# Le corpus canonique (training_data_final_12gb.jsonl) est PROTÉGÉ :
# si le fichier existe déjà, on l'utilise tel quel (reproductibilité).
# Pour régénérer depuis les APIs : supprimez le fichier puis relancez cette cellule.

corpus_file = FT_DIR / "training_data_final_12gb.jsonl"

if corpus_file.exists():
    with open(corpus_file, 'r', encoding='utf-8') as _f:
        n_existing = sum(1 for _ in _f)
    print(f"✅ Corpus déjà présent : {n_existing} paires → {corpus_file}")
    print("   (Pour régénérer : corpus_file.unlink() puis relancez la cellule)")
else:
    # Cible : ~5,400 paires / ~540K tokens pour tenir en 12 GB
    MAX_TOKENS = 550_000
    MAX_PAIRS = 6_000

    # Melanger pour diversite
    random.shuffle(unique_pairs)

    # Selectionner en respectant la limite de tokens
    selected = []
    total_tokens = 0
    for pair in unique_pairs:
        tokens_est = len(pair['prompt'] + pair['completion']) // 4  # estimation ~4 chars/token
        if total_tokens + tokens_est > MAX_TOKENS or len(selected) >= MAX_PAIRS:
            break
        selected.append(pair)
        total_tokens += tokens_est

    with open(corpus_file, 'w', encoding='utf-8') as f:
        for pair in selected:
            f.write(json.dumps(pair, ensure_ascii=False) + '\n')

    print(f"Corpus final : {len(selected)} paires")
    print(f"Tokens estimes : {total_tokens:,}")
    print(f"Fichier : {corpus_file}")
    print(f"Taille : {corpus_file.stat().st_size / 1024**2:.2f} MB")

Corpus final : 5118 paires
Tokens estimes : 549,948
Fichier : D:\Projets\ADS\data\fine_tuning\training_data_final_12gb.jsonl
Taille : 2.41 MB


## 6. Fine-tuning QLoRA

Configuration :
- **Quantisation** : QLoRA NF4 4-bit + double quantisation
- **LoRA** : r=32, alpha=64, dropout=0.05
- **Modules cibles** : q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- **Training** : batch_size=1, gradient_accumulation=8, lr=2e-4, cosine scheduler
- **Early stopping** : patience=2, threshold=0.01

In [ ]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig, EarlyStoppingCallback,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset

# --- 6.1 Charger et splitter les donnees ---
data = []
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

np.random.seed(42)
indices = np.random.permutation(len(data))
data = [data[i] for i in indices]

n_total = len(data)
n_test = int(n_total * 0.10)
n_val = int(n_total * 0.10)
n_train = n_total - n_test - n_val

train_data = data[:n_train]
val_data = data[n_train:n_train+n_val]
test_data = data[n_train+n_val:]

print(f"Train : {n_train} | Val : {n_val} | Test : {n_test}")

In [ ]:
# --- 6.2 Charger modele avec QLoRA ---
print("Chargement tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("Chargement modele (QLoRA 4-bit NF4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

print(f"Modele charge. VRAM : {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# --- 6.3 Configuration LoRA ---
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())
print(f"Parametres entrainables : {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"Parametres total : {total_p:,}")

In [ ]:
# --- 6.4 Tokenization ---
MAX_LENGTH = 512

def format_instruction(prompt, completion):
    return f"<s>[INST] {SYSTEM_PROMPT}\n\nQuestion : {prompt} [/INST] {completion}</s>"

def tokenize_data(data_list):
    texts = [format_instruction(item['prompt'], item['completion']) for item in data_list]
    encodings = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                          padding='max_length', return_tensors='pt')
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': encodings['input_ids'].clone()
    })

print("Tokenization train...")
train_dataset = tokenize_data(train_data)
print("Tokenization validation...")
val_dataset = tokenize_data(val_data)

print(f"Train : {len(train_dataset)} samples | Val : {len(val_dataset)} samples")

In [ ]:
# --- 6.5 Training ---
class VRAMCallback(TrainerCallback):
    def __init__(self):
        self.vram_peak = 0
    def on_step_end(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            current = torch.cuda.memory_allocated() / 1024**3
            if current > self.vram_peak:
                self.vram_peak = current

OUTPUT_DIR = ADAPTERS_DIR / "mistral-7b-phase1_validation"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    per_device_eval_batch_size=1,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='paged_adamw_8bit',
    max_grad_norm=0.3,
    seed=42,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

vram_cb = VRAMCallback()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[
        vram_cb,
        EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.01)
    ]
)

print("Demarrage du fine-tuning...")
start_time = datetime.now()
result = trainer.train()
duration = (datetime.now() - start_time).total_seconds() / 60

print(f"\n--- TRAINING TERMINE ---")
print(f"Duree : {duration:.1f} min")
print(f"Loss finale : {result.training_loss:.4f}")
print(f"VRAM peak : {vram_cb.vram_peak:.2f} GB")

In [ ]:
# --- 6.6 Sauvegarder le modele ---
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Metadonnees
metadata = {
    "model": MODEL_NAME,
    "method": "QLoRA NF4 4-bit",
    "lora": {"r": 32, "alpha": 64, "dropout": 0.05},
    "corpus": {"num_examples": len(train_data), "max_length": MAX_LENGTH},
    "training": {
        "duration_minutes": round(duration, 1),
        "final_loss": round(result.training_loss, 4),
        "vram_peak_gb": round(vram_cb.vram_peak, 2)
    },
    "timestamp": datetime.now().isoformat()
}
with open(OUTPUT_DIR / "training_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Modele sauvegarde dans : {OUTPUT_DIR}")

## 7. Evaluation

Metriques evaluees :
- **Perplexite** : mesure la qualite predictive du modele
- **Precision factuelle** : 30 questions avec seuil de 80% de mots-cles
- **Garde-fous** : 7 questions hors-perimetre
- **Vitesse** : tokens par seconde

In [ ]:
# --- 7.1 Perplexite sur le set de test ---
print("Calcul perplexite sur le set de test...")
test_dataset = tokenize_data(test_data)

model.eval()
total_loss = 0
n_batches = 0

with torch.no_grad():
    for i in range(min(len(test_dataset), 200)):
        input_ids = test_dataset[i]['input_ids'].unsqueeze(0).to(model.device)
        labels = test_dataset[i]['labels'].unsqueeze(0).to(model.device)
        attention_mask = test_dataset[i]['attention_mask'].unsqueeze(0).to(model.device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
        n_batches += 1

avg_loss = total_loss / n_batches
perplexity = np.exp(avg_loss)

print(f"Loss moyenne : {avg_loss:.4f}")
print(f"Perplexite : {perplexity:.4f}")

In [ ]:
# --- 7.2 Precision factuelle ---
# Chargement des questions depuis test_questions.json (20 questions)
test_questions_file = PROJECT_ROOT / "data" / "test_questions.json"
if test_questions_file.exists():
    with open(test_questions_file, 'r', encoding='utf-8') as f:
        EVAL_QUESTIONS = json.load(f)
    print(f"Questions chargees depuis {test_questions_file.name} : {len(EVAL_QUESTIONS)}")
else:
    print(f"ATTENTION: {test_questions_file} introuvable, utilisation de questions par defaut")
    EVAL_QUESTIONS = [
        {"question": "Quel est le montant du marche de voirie a Montpellier en 2024 ?",
         "keywords": ["montpellier", "voirie", "montant"]},
        {"question": "Quelle procedure pour un marche de 35 000 EUR ?",
         "keywords": ["mapa", "adaptee", "seuil"]},
        {"question": "Quel est le role du DECP ?",
         "keywords": ["commande", "publique", "donnees", "transparence"]},
        {"question": "Quelle est la meteo a Tokyo ?",
         "keywords": ["hors", "disponible", "territoire", "corpus"], "should_refuse": True},
    ]

def generate_response(prompt, max_new_tokens=256):
    formatted = f"<s>[INST] {SYSTEM_PROMPT}\n\nQuestion : {prompt} [/INST]"
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if '[/INST]' in text:
        return text.split('[/INST]')[-1].strip()
    return text[len(formatted):].strip()

print(f"\nEvaluation sur {len(EVAL_QUESTIONS)} questions...\n")
correct = 0
guardrails_ok = 0
n_factuel = 0
n_guardrail = 0

for q in EVAL_QUESTIONS:
    response = generate_response(q['question'])
    response_lower = response.lower()

    matched = sum(1 for kw in q['keywords'] if kw.lower() in response_lower)
    score = matched / len(q['keywords'])
    is_correct = score >= 0.80

    is_guardrail = q.get('should_refuse', False)

    if is_guardrail:
        n_guardrail += 1
        if is_correct:
            guardrails_ok += 1
    else:
        n_factuel += 1
        if is_correct:
            correct += 1

    status = "OK" if is_correct else "FAIL"
    q_type = "GUARDRAIL" if is_guardrail else "FACTUEL"
    print(f"[{status}] [{q_type}] {q['question'][:65]}... ({score:.0%})")

print(f"\n--- RESULTATS ---")
if n_factuel > 0:
    print(f"Precision factuelle : {correct}/{n_factuel} ({correct/n_factuel*100:.1f}%)")
if n_guardrail > 0:
    print(f"Garde-fous : {guardrails_ok}/{n_guardrail} ({guardrails_ok/n_guardrail*100:.1f}%)")

In [ ]:
# --- 7.3 Vitesse d'inference ---
import time

test_prompt = "Quels sont les marches publics de Montpellier en 2024 ?"
formatted = f"<s>[INST] {SYSTEM_PROMPT}\n\nQuestion : {test_prompt} [/INST]"
inputs = tokenizer(formatted, return_tensors='pt').to(model.device)

# Warmup
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

# Mesurer
n_runs = 3
total_tokens = 0
total_time = 0
for _ in range(n_runs):
    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                            temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - start
    n_tokens = out.shape[1] - inputs['input_ids'].shape[1]
    total_tokens += n_tokens
    total_time += elapsed

speed = total_tokens / total_time
print(f"Vitesse : {speed:.1f} tokens/seconde")

In [ ]:
# --- 7.4 Resume complet ---
results_summary = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL_NAME,
    "method": "QLoRA NF4 4-bit",
    "lora": {"r": 32, "alpha": 64, "dropout": 0.05},
    "perplexity": round(perplexity, 4),
    "loss": round(avg_loss, 4),
    "accuracy": f"{correct}/{n_factuel}" if n_factuel > 0 else "N/A",
    "accuracy_pct": round(correct / n_factuel * 100, 1) if n_factuel > 0 else 0,
    "guardrails": f"{guardrails_ok}/{n_guardrail}" if n_guardrail > 0 else "N/A",
    "guardrails_pct": round(guardrails_ok / n_guardrail * 100, 1) if n_guardrail > 0 else 0,
    "speed_tokens_per_sec": round(speed, 1),
    "training_duration_min": round(duration, 1),
    "vram_peak_gb": round(vram_cb.vram_peak, 2),
    "corpus_train": len(train_data),
    "corpus_val": len(val_data),
    "corpus_test": len(test_data),
    "eval_questions_count": len(EVAL_QUESTIONS)
}

print("\n" + "=" * 60)
print("RESUME EVALUATION")
print("=" * 60)
print(f"  Perplexite       : {results_summary['perplexity']}")
print(f"  Loss             : {results_summary['loss']}")
print(f"  Precision        : {results_summary['accuracy']} ({results_summary['accuracy_pct']}%)")
print(f"  Garde-fous       : {results_summary['guardrails']} ({results_summary['guardrails_pct']}%)")
print(f"  Vitesse          : {results_summary['speed_tokens_per_sec']} t/s")
print(f"  Duree training   : {results_summary['training_duration_min']} min")
print(f"  VRAM peak        : {results_summary['vram_peak_gb']} GB")
print(f"  Corpus           : {results_summary['corpus_train']} train / {results_summary['corpus_val']} val / {results_summary['corpus_test']} test")
print("=" * 60)

# Sauvegarder les resultats
benchmarks_dir = RESULTS_DIR / "benchmarks"
benchmarks_dir.mkdir(parents=True, exist_ok=True)
results_file = benchmarks_dir / f"notebook_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print(f"\nResultats sauvegardes : {results_file}")

## 8. Test interactif

Posez vos questions au modele fine-tune !

In [ ]:
# --- Mode interactif : modifier la question et relancer la cellule ---
question = "Quels sont les marches publics de Montpellier en 2024 ?"

print(f"Q: {question}\n")
response = generate_response(question, max_new_tokens=512)
print(f"R: {response}")

In [ ]:
# --- Comparaison base vs fine-tune (necessite rechargement du modele base) ---
# Decommentez pour comparer :

# from peft import PeftModel
#
# # Charger modele base seul
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
# )
# base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# base_tokenizer.pad_token = base_tokenizer.eos_token
#
# question = "Quels departements sont couverts par les donnees DECP ?"
#
# # Reponse base
# formatted = f"<s>[INST] {SYSTEM_PROMPT}\n\nQuestion : {question} [/INST]"
# inputs = base_tokenizer(formatted, return_tensors='pt').to(base_model.device)
# with torch.no_grad():
#     out = base_model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
# base_response = base_tokenizer.decode(out[0], skip_special_tokens=True)
#
# # Reponse fine-tune
# ft_response = generate_response(question)
#
# print("=== MODELE BASE ===")
# print(base_response.split('[/INST]')[-1].strip())
# print("\n=== MODELE FINE-TUNE ===")
# print(ft_response)

print("Decommentez le code ci-dessus pour comparer base vs fine-tune.")

## 9. Sauvegarde finale et nettoyage

Le modele fine-tune est sauvegarde dans `models/adapters/mistral-7b-phase1_validation/`.  
Pour le reutiliser plus tard :

In [ ]:
# --- Rechargement du modele fine-tune (pour utilisation ulterieure) ---
# from peft import PeftModel
#
# adapter_path = str(ADAPTERS_DIR / "mistral-7b-phase1_validation")
#
# # Charger base + adapters
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=BitsAndBytesConfig(
#         load_in_4bit=True, bnb_4bit_use_double_quant=True,
#         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
#     ),
#     device_map="auto"
# )
# model = PeftModel.from_pretrained(model, adapter_path)
# tokenizer = AutoTokenizer.from_pretrained(adapter_path)
#
# print(f"Modele recharge depuis {adapter_path}")

print("Pipeline complet termine !")
print(f"Adaptateurs : {OUTPUT_DIR}")
print(f"Corpus : {corpus_file}")